# 04b — Per-session normalized features

Produces a third feature CSV alongside `features_common_no_baro.csv` (standard) and `features_common_no_baro_extended.csv` (standard + event burstiness features). This one applies per-session amplitude normalization before computing features.

**Why this exists.** Notebook 07 showed that the LOSO walking F1 collapsed from 0.96 (session split) to 0.72 (LOSO). The drop comes from participant-level amplitude differences in the linear accelerometer (Babis's walking magnitude is roughly twice Athina's and Xiomara's, based on phone placement). When any extreme of the amplitude distribution is held out, the model trained on the remaining users cannot extrapolate.

Per-session normalization divides each session's linear accelerometer (and gyroscope) by its own RMS. This removes amplitude scale entirely while preserving cadence, frequency, and shape information. Walking from any participant at any phone placement produces similar normalized signals.

**Why a separate CSV.** Normalization removes absolute amplitude, which is exactly the signal that distinguishes stationary from low-motion vehicular modes. The hierarchical model in notebook 08 will use normalized features at Level 2A (walking vs cycling) and absolute features at Level 2B (stationary vs vehicle). Producing two CSVs keeps each level's input clean.

**Output.** `data/processed/features_common_no_baro_normalized.csv`. Same rows and metadata as the standard CSV. Feature columns are prefixed `norm_` so the two files can be merged on metadata without column collisions.

## 1. Imports

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import features as F
from src.data_loading import build_session_index

INTERIM_DIR   = PROJECT_ROOT / 'data' / 'interim'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 2. Confirm the normalization function

Quick sanity check on one session to verify the normalization works as expected. After normalization the linacc magnitude RMS should equal 1.0 exactly.

In [2]:
session_index = build_session_index(INTERIM_DIR)
print(f'Found {len(session_index)} sessions')

# Pick one session and verify normalization
test_sess = session_index[0]
df_raw = pd.read_csv(test_sess['path'])
df_norm = F.per_session_normalize(df_raw)

raw_linacc_mag = np.sqrt(df_raw['lin_ax']**2 + df_raw['lin_ay']**2 + df_raw['lin_az']**2)
norm_linacc_mag = np.sqrt(df_norm['lin_ax']**2 + df_norm['lin_ay']**2 + df_norm['lin_az']**2)

print(f'Test session: {test_sess["session_id"]}')
print(f'Raw  linacc_mag RMS: {np.sqrt(np.mean(raw_linacc_mag**2)):.4f}')
print(f'Norm linacc_mag RMS: {np.sqrt(np.mean(norm_linacc_mag**2)):.4f}  (should be 1.0)')

assert abs(np.sqrt(np.mean(norm_linacc_mag**2)) - 1.0) < 1e-6

Found 56 sessions
Test session: athina_bus_day1
Raw  linacc_mag RMS: 0.7635
Norm linacc_mag RMS: 1.0000  (should be 1.0)


## 3. Featurise every session with normalization

Same windowing as the standard CSV, but `apply_session_normalization=True` divides each session's signals by their RMS before windowing and feature extraction. Run time should be similar to notebook 04 section 5.

In [3]:
all_features_norm = []
for sess in session_index:
    meta = {
        'session_id':  sess['session_id'],
        'participant': sess['participant'],
        'mode':        sess['mode'],
        'mode_id':     F.MODE_TO_ID.get(sess['mode'], -1),
        'day':         sess['day'],
    }
    # Skip non-session files (those where parse_session_id returned None)
    if sess['mode'] not in F.MODES:
        continue
    feat_rows, _ = F.featurise_session(
        Path(sess['path']), meta,
        include_event_features=False,
        apply_session_normalization=True,
    )
    all_features_norm.extend(feat_rows)

features_norm_df = pd.DataFrame(all_features_norm)
print(f'Total windows: {len(features_norm_df)}')
print(f'Total columns: {len(features_norm_df.columns)}')

Total windows: 14513
Total columns: 138


## 4. Prefix feature columns with `norm_`

Metadata columns keep their original names so the file can be joined with the standard CSV on the session and window identifiers. Feature columns are renamed so the normalized variant of `lin_ay_iqr` becomes `norm_lin_ay_iqr` and so on.

In [4]:
meta_cols = ['session_id', 'participant', 'mode', 'mode_id', 'day',
             'window_index', 'window_start_s', 'window_end_s', 'flagged_frac']

feature_cols = [c for c in features_norm_df.columns if c not in meta_cols]
print(f'Feature columns: {len(feature_cols)}')

# Replace NaN and inf before renaming
features_norm_df[feature_cols] = features_norm_df[feature_cols].fillna(0.0)
features_norm_df[feature_cols] = features_norm_df[feature_cols].replace(
    [np.inf, -np.inf], 0.0
)

rename_dict = {c: f'norm_{c}' for c in feature_cols}
features_norm_df = features_norm_df.rename(columns=rename_dict)
norm_feature_cols = [f'norm_{c}' for c in feature_cols]

features_norm_df = features_norm_df[meta_cols + norm_feature_cols]
print(f'After renaming: {len(features_norm_df.columns)} cols ',
      f'({len(meta_cols)} metadata + {len(norm_feature_cols)} features)')

Feature columns: 129
After renaming: 138 cols  (9 metadata + 129 features)


## 5. Quick sanity check on normalized features

Verify that the normalization removed amplitude differences between participants while preserving cadence. The standard `lin_ay_rms` should differ ~2x between Babis and the others; the normalized `norm_lin_ay_rms` should be much closer.

In [5]:
# Compare with standard features
std_path = PROCESSED_DIR / 'features_common_no_baro.csv'
if std_path.exists():
    std_df = pd.read_csv(std_path)

    print('Median linacc_mag_rms per participant (standard features):')
    print(std_df.groupby('participant')['linacc_mag_rms'].median().round(3).to_string())
    print()
    print('Median norm_linacc_mag_rms per participant (normalized features):')
    print(features_norm_df.groupby('participant')['norm_linacc_mag_rms']
          .median().round(3).to_string())
else:
    print('Standard features CSV not found; skipping comparison.')
    print()
    print('Median norm_linacc_mag_rms per participant:')
    print(features_norm_df.groupby('participant')['norm_linacc_mag_rms']
          .median().round(3).to_string())

Median linacc_mag_rms per participant (standard features):
participant
athina     0.334
babis      0.678
xiomara    0.297

Median norm_linacc_mag_rms per participant (normalized features):
participant
athina     0.770
babis      0.878
xiomara    0.773


## 6. Save

In [6]:
out_path = PROCESSED_DIR / 'features_common_no_baro_normalized.csv'
features_norm_df.to_csv(out_path, index=False)
print(f'Saved {len(features_norm_df)} rows x {len(features_norm_df.columns)} cols to {out_path}')

Saved 14513 rows x 138 cols to C:\Projects\this project-transport-mode\data\processed\features_common_no_baro_normalized.csv


## 7. Summary

Three feature CSVs now exist in `data/processed/`:

1. `features_common_no_baro.csv`: standard features (the LOSO winner from notebook 05 used this).
2. `features_common_no_baro_extended.csv`: standard features + 12 event burstiness features. Use for Level 2B in notebook 08.
3. `features_common_no_baro_normalized.csv`: features computed on per-session normalized signals. Column prefix is `norm_`. Use for Level 2A in notebook 08.

All three files have identical row counts and metadata columns. Notebook 08 will merge them on `(session_id, window_index)` to select the right feature representation per level.